# SChain INGEST

Process a file or directory already available in Colab. INGEST does not search the web or
expand the frontier; it runs documents through extraction, resolution, validation and export.

In [ ]:
# Run settings
INPUT_PATH = "/content/documents"
CONFIG_PATH = "config/ingest.yaml"
SAVE_TO_NEO4J = False
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
BRANCH = "mirza"
# Bounded context and eager execution reduce GPU memory pressure in Colab.
MAX_MODEL_LEN = 16384
MODEL_START_TIMEOUT_SECONDS = 1200


## Install SChain

In [ ]:
import os
import pathlib
import subprocess
import sys

REPO_DIR = pathlib.Path("/content/SChain")
if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch",
         "https://github.com/ErfanBabakhani/Schain.git", str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "vllm"], check=True)

## Optional Neo4j credentials

In [ ]:
if SAVE_TO_NEO4J:
    from google.colab import userdata

    for name in ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"):
        value = userdata.get(name)
        if not value:
            raise RuntimeError(f"Missing Colab secret: {name}")
        os.environ[name] = value

## Start the local model server

Select **Runtime > Change runtime type > GPU** first. No LLM API key is needed for this local model. Startup checks the GPU, shows progress and reports the server log on failure. A first model download can take several minutes.

In [ ]:
import httpx
import time

VLLM_URL = "http://127.0.0.1:8000/v1"
LOG_PATH = pathlib.Path("/content/vllm.log")

def vllm_ready() -> bool:
    try:
        response = httpx.get(f"{VLLM_URL}/models", timeout=2)
        response.raise_for_status()
        models = [item["id"] for item in response.json()["data"]]
        if MODEL not in models:
            raise RuntimeError(f"Port 8000 serves {models}, expected {MODEL}. Stop the old server or restart the runtime.")
        return True
    except httpx.HTTPError:
        return False

def server_error(message):
    tail = "\n".join(LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()[-80:])
    return RuntimeError(f"{message}\nLast vLLM log lines:\n{tail}")

if not vllm_ready():
    # Probe in a separate process: pip may have replaced torch in this notebook kernel.
    probe = subprocess.run(
        [sys.executable, "-c", "import torch; print('torch:', torch.__version__); "
         "assert torch.cuda.is_available(), 'No CUDA GPU. Select Runtime > Change runtime type > GPU.'; "
         "print('GPU:', torch.cuda.get_device_name(0)); "
         "print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))"],
        capture_output=True, text=True,
    )
    print(probe.stdout)
    if probe.returncode:
        raise RuntimeError(f"GPU preflight failed:\n{probe.stderr}")
    command = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL, "--host", "127.0.0.1", "--port", "8000",
        "--dtype", "half", "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", "0.90", "--max-num-seqs", "1",
        "--enforce-eager",
    ]
    with LOG_PATH.open("w", encoding="utf-8") as log:
        vllm_process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT)
    print(f"Starting {MODEL}; log: {LOG_PATH}", flush=True)
    deadline = time.monotonic() + MODEL_START_TIMEOUT_SECONDS
    try:
        while time.monotonic() < deadline:
            if vllm_process.poll() is not None:
                raise server_error(f"vLLM exited with code {vllm_process.returncode}.")
            if vllm_ready():
                break
            print("Waiting for model download / startup...", flush=True)
            time.sleep(10)
        else:
            raise server_error("vLLM startup timed out.")
    except BaseException:
        if vllm_process.poll() is None:
            vllm_process.terminate()
            try:
                vllm_process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                vllm_process.kill()
                vllm_process.wait()
        raise

response = httpx.post(
    f"{VLLM_URL}/chat/completions",
    json={"model": MODEL, "messages": [{"role": "user", "content": "Reply OK."}], "max_tokens": 8},
    timeout=120,
)
if response.is_error:
    raise RuntimeError(f"Model inference failed ({response.status_code}): {response.text[:2000]}")
print("LLM ready:", response.json()["choices"][0]["message"]["content"])


## Run INGEST and export the graph

In [ ]:
from schain.config import load_config
import yaml

# Pass the selected model and endpoint to both CLI subprocesses.
runtime_config = load_config(CONFIG_PATH)
runtime_config.setdefault("llm", {}).update({"provider": "local", "model": MODEL, "base_url": VLLM_URL})
runtime_path = pathlib.Path("/content/schain-ingest-runtime.yaml")
runtime_path.write_text(yaml.safe_dump(runtime_config, sort_keys=False), encoding="utf-8")

command = [
    sys.executable, "-m", "schain", "ingest",
    "--config", str(runtime_path),
    "--input", INPUT_PATH,
    "--resume",
    "--save" if SAVE_TO_NEO4J else "--no-save",
]
subprocess.run(command, check=True)

subprocess.run([
    sys.executable, "-m", "schain", "export", "--config", str(runtime_path),
], check=True)

## Inspect generated artifacts

In [ ]:
import json

for path in sorted(pathlib.Path("outputs").glob("*")):
    print(path)